# Supplementary analyses

Supporting evidence not included in the three-page letter. Each cell is
independent and reads from the same staged inputs as the main notebook.


## Spatial autocorrelation range

Semivariogram of night LST. Establishes a range of 10–12 km against 1 km node spacing.

In [ ]:
# ============================================================
# TEST B — SPATIAL AUTOCORRELATION RANGE OF NIGHT LST
# ------------------------------------------------------------
# STANDALONE. Mounts Drive, reads the CSVs and geojsons directly,
# trains nothing. Safe on a cold kernel.
#
# WHY THIS TEST
# NoGraph beat Full on RMSE in 5 of 6 configs at 1 km spacing. On the
# Landsat data at 500 m the ordering was the opposite: skip-GAT 0.68
# against OLS 0.60. So message passing helped at 500 m and does not at
# 1 km.
#
# The hypothesis is about SCALE: a k-NN graph is only useful if a
# node's neighbours sit inside the distance over which the field stays
# correlated. This measures that distance from the DATA, with no model
# involved, so it can explain the model results rather than restate
# them.
#
# A semivariogram plots half the mean squared difference between node
# pairs against their separation. It rises with distance and flattens
# at the SILL; the distance where it flattens is the RANGE. Beyond the
# range, two nodes tell you nothing about each other, and averaging
# over them destroys signal instead of adding any.
#
# The decisive comparison is RANGE against the actual distance a k-NN
# neighbourhood reaches, which is computed here rather than assumed.
# ============================================================
import os
import shutil
import numpy as np
import pandas as pd
import geopandas as gpd

# ---- mount + stage ------------------------------------------
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print(f"(drive mount skipped: {e})")

DRIVE_DATA = '/content/drive/MyDrive/UHI'
LOCAL_DATA = '/content/data_modis'
SUFFIX = '_60d'          # set to '_v2' for the 14-day extraction

DATASETS = [
    ('Bangalore', 'Blr', 'April',    f'Blr_April_MODIS{SUFFIX}.csv',    'Blr_Nodes_1km.geojson'),
    ('Bangalore', 'Blr', 'December', f'Blr_December_MODIS{SUFFIX}.csv', 'Blr_Nodes_1km.geojson'),
    ('Hyderabad', 'Hyd', 'April',    f'Hyd_April_MODIS{SUFFIX}.csv',    'Hyd_Nodes_1km.geojson'),
    ('Hyderabad', 'Hyd', 'December', f'Hyd_December_MODIS{SUFFIX}.csv', 'Hyd_Nodes_1km.geojson'),
    ('Delhi',     'Dlh', 'April',    f'Dlh_April_MODIS{SUFFIX}.csv',    'Dlh_Nodes_1km.geojson'),
    ('Delhi',     'Dlh', 'December', f'Dlh_December_MODIS{SUFFIX}.csv', 'Dlh_Nodes_1km.geojson'),
]

os.makedirs(LOCAL_DATA, exist_ok=True)
for name in sorted({f for d in DATASETS for f in d[3:5]}):
    src, dst = f"{DRIVE_DATA}/{name}", f"{LOCAL_DATA}/{name}"
    if os.path.exists(dst) and os.path.exists(src) \
            and os.path.getsize(dst) == os.path.getsize(src):
        continue
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"  staged {name}")
    else:
        print(f"  [MISSING] {name}")

ID_COL, TIME_COL, TARGET = 'id', 'timestamp', 'lst_modis'
K_GRID = [2, 4, 8, 16]
MAX_PAIRS_NODES = 700       # subsample for the O(n^2) distance matrix


def align_ids(s):
    try:
        return s.astype('int64')
    except (ValueError, TypeError):
        return s.astype(str)


def load_nodes(path):
    """GeoJSON carries no CRS member, so geopandas assumes 4326. These
    were built with coveringGrid in EPSG:32643 and hold METRES;
    to_crs() on that returns inf. Magnitude is the unambiguous test."""
    g = gpd.read_file(path)
    b = g.total_bounds
    if max(abs(b[0]), abs(b[2]), abs(b[1]), abs(b[3])) > 180:
        g = g.set_crs(32643, allow_override=True)
    else:
        if g.crs is None:
            g = g.set_crs(4326)
        g = g.to_crs(epsg=32643)
    return g


def semivariogram(pos, field, edges):
    """Half the mean squared difference between node pairs, binned by
    separation distance."""
    d = np.sqrt(((pos[:, None, :] - pos[None, :, :]) ** 2).sum(-1))
    s = 0.5 * (field[:, None] - field[None, :]) ** 2
    iu = np.triu_indices(len(pos), k=1)
    d, s = d[iu], s[iu]
    out = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (d >= lo) & (d < hi)
        if m.sum() >= 30:
            out.append((0.5 * (lo + hi), float(s[m].mean()), int(m.sum())))
    return out


summary = []
for city, short, month, csvname, geoname in DATASETS:
    csv_p, geo_p = f"{LOCAL_DATA}/{csvname}", f"{LOCAL_DATA}/{geoname}"
    if not (os.path.exists(csv_p) and os.path.exists(geo_p)):
        print(f"\n{short} {month}: files missing, skipping")
        continue

    print("\n" + "=" * 72)
    print(f"{city} — {month}")
    print("=" * 72)

    df = pd.read_csv(csv_p)
    df[ID_COL] = align_ids(df[ID_COL])
    if np.issubdtype(df[TIME_COL].dtype, np.number):
        df[TIME_COL] = pd.to_datetime(df[TIME_COL], unit='ms')
    else:
        df[TIME_COL] = pd.to_datetime(df[TIME_COL])
    df = df[df['pass'].str.contains('night', na=False)]
    df = df[df[TARGET].notna()]

    gdf = load_nodes(geo_p)
    gdf[ID_COL] = align_ids(gdf[ID_COL])
    gdf = gdf[gdf[ID_COL].isin(df[ID_COL].unique())].sort_values(ID_COL)
    node_ids = gdf[ID_COL].values
    pos_all = np.column_stack([gdf.geometry.centroid.x.values,
                               gdf.geometry.centroid.y.values])

    # persistent spatial pattern: each night's field mean removed,
    # then averaged over nights. This is the structure the model is
    # asked to reproduce.
    df['anom'] = df[TARGET] - df.groupby(TIME_COL)[TARGET].transform('mean')
    field_all = df.groupby(ID_COL)['anom'].mean().reindex(node_ids).values

    ok = np.isfinite(field_all)
    pos_all, field_all = pos_all[ok], field_all[ok]
    n = len(pos_all)
    print(f"  {n} nodes | {df[TIME_COL].nunique()} night passes | "
          f"persistent anomaly sd {field_all.std():.2f} °C")

    # --- how far does a k-NN neighbourhood actually reach? ---------
    sub = np.random.default_rng(0).choice(
        n, size=min(n, MAX_PAIRS_NODES), replace=False)
    P, F = pos_all[sub], field_all[sub]
    D = np.sqrt(((pos_all[:, None, :] - pos_all[None, :, :]) ** 2).sum(-1)) \
        if n <= 1200 else None
    if D is not None:
        np.fill_diagonal(D, np.inf)
        reach = {k: float(np.median(np.sort(D, axis=1)[:, k - 1]))
                 for k in K_GRID}
        print("  median distance to the k-th nearest neighbour:")
        print("    " + "   ".join(f"k={k}: {v:,.0f} m" for k, v in reach.items()))
    else:
        reach = {}

    # --- semivariogram, fine near the origin ----------------------
    edges = np.concatenate([np.arange(0, 3001, 500),
                            np.arange(4000, 15001, 1000)])
    sv = semivariogram(P, F, edges)
    if len(sv) < 4:
        print("  too few pairs for a variogram")
        continue

    lags = np.array([a for a, _, _ in sv])
    vals = np.array([b for _, b, _ in sv])
    sill = float(np.mean(vals[-3:]))
    print(f"\n  {'lag (m)':>10}{'semivariance':>15}{'% of sill':>11}{'pairs':>9}")
    for a, b, c in sv:
        print(f"  {a:>10,.0f}{b:>15.4f}{100*b/sill:>10.0f}%{c:>9,}")

    # range: first lag reaching 95% of the sill
    idx = np.where(vals >= 0.95 * sill)[0]
    rng_m = float(lags[idx[0]]) if len(idx) else float('nan')
    print(f"\n  sill {sill:.4f}  |  range (95% of sill) ~{rng_m:,.0f} m")

    verdict = ''
    if reach and np.isfinite(rng_m):
        r8 = reach.get(8, np.nan)
        ratio = r8 / rng_m if rng_m > 0 else np.nan
        print(f"  k=8 reaches {r8:,.0f} m, i.e. {ratio:.2f}x the range")
        if ratio >= 1.0:
            verdict = 'k=8 EXCEEDS range'
            print("  -> the neighbourhood extends past the distance over which")
            print("     the field stays correlated, so message passing averages")
            print("     nodes that carry no information about each other.")
        elif ratio >= 0.6:
            verdict = 'k=8 near range'
            print("  -> the neighbourhood sits near the edge of the correlated")
            print("     zone; the outer neighbours contribute little.")
        else:
            verdict = 'k=8 inside range'
            print("  -> neighbours sit well inside the correlated zone and")
            print("     should carry information. If the graph still does not")
            print("     help, that is an architecture problem, not scale.")

    summary.append(dict(config=f"{short} {month}", nodes=n,
                        anom_sd=field_all.std(), sill=sill, range_m=rng_m,
                        reach_k8=reach.get(8, np.nan), verdict=verdict))

print("\n" + "=" * 72)
print("SUMMARY")
print("=" * 72)
s = pd.DataFrame(summary)
if len(s):
    print(s.round(3).to_string(index=False))
    print(f"""
  Node spacing here is 1,000 m. The Landsat experiment, where the graph
  DID help, used 500 m nodes -- so its k=8 neighbourhood reached about
  half as far.

  If range comes out near or below the k=8 reach, the two experiments
  are explained by one mechanism: message passing helps only when node
  spacing is well inside the field's correlation range. That is a
  scale result about when graph methods suit spatial fields, and it is
  a more useful contribution than either experiment on its own.

  If range is comfortably larger than the k=8 reach, the neighbours do
  carry information and the model is failing to use it -- which points
  at architecture instead, and the k-sweep would then be worth the
  compute.
""")
else:
    print("  nothing computed -- check the staged filenames above.")

## Does message passing add anything? (k sweep)

In [ ]:
# ============================================================
# DOES MESSAGE PASSING ADD ANYTHING? — focused test
# ------------------------------------------------------------
# The semivariogram settled one question and opened another.
# Neighbours at the k=8 distance (1,414 m) differ from the centre node
# by only 5-12% of total field variance, and the range is 10-12 km.
# So neighbours are highly informative and the "1 km spacing is past
# the correlation range" hypothesis is dead.
#
# WHICH LEAVES: the model has informative neighbours and does not
# appear to benefit. Two explanations, and they call for different
# responses.
#
#   (a) REDUNDANCY. The morphological features are themselves smooth
#       across the city, so a per-node model reading bld_cover,
#       bld_height and NDVI already recovers most of what
#       neighbour-averaging would supply. Message passing over
#       features then has nothing left to add. No architecture fixes
#       this -- it is a property of the data.
#
#   (b) WIRING. The graph is there but the model is not using it,
#       which would be a genuine architecture problem worth one round
#       of investigation.
#
# This runs the SAME PhyRes model at k = 0, 4, 8, 16 on the
# best-covered config, three seeds each. The earlier Full-vs-NoGraph
# comparison confounded message passing with encoder type and
# parameter count; here only connectivity changes.
#
# It also measures feature smoothness directly, which is what
# separates (a) from (b): if the features are as spatially smooth as
# the target, redundancy is the explanation.
#
# 12 runs, roughly 10-15 minutes.
# ============================================================
import numpy as np
import pandas as pd
import torch
from scipy import stats
import torch.nn.functional as F

# Delhi April: 108 night passes, 873 nodes, best coverage of the six
CFG = next(c for c in CONFIGS if c['city'] == 'Delhi' and c['month'] == 'April')
K_GRID = [0, 4, 8, 16]
SEEDS = [10, 11, 12]

D = prepare(CFG, HP)
ds0 = D['pruned']['train'].dataset
pos = ds0.pos


# ============================================================
# PART 1 — how much do neighbours add BEYOND the node's own features?
# ============================================================
# If a node's features already predict its neighbours' features, then
# the neighbourhood mean carries little the node does not have, and
# message passing is redundant regardless of architecture.
print("=" * 74)
print("PART 1 — ARE THE FEATURES ALREADY SPATIALLY SMOOTH?")
print("=" * 74)

x_raw = ds0.x_raw[:, -1, :].cpu().numpy()          # physical units, last step
y_node = ds0.y.cpu().numpy()
y_anom = (y_node - y_node.mean(axis=0, keepdims=True)).mean(axis=1)
ei8 = knn_graph(pos, k=8, loop=False).cpu().numpy()
nbr = ei8[0].reshape(-1, 8)

names = ['t2m', 'u10', 'v10', 'ssr', 'str', 'slhf',
         'NDVI', 'NDBI', 'bld_height', 'bld_cover', 'no2']
print(f"  {'feature':<12}{'corr(own, nbr mean)':>22}")
for i in (6, 7, 8, 9, 10):
    own = x_raw[:, i]
    nb = x_raw[nbr, i].mean(axis=1)
    if own.std() > 1e-9 and nb.std() > 1e-9:
        print(f"  {names[i]:<12}{np.corrcoef(own, nb)[0, 1]:>22.3f}")
print("""
  Values near 1 mean a node's own feature already tells you its
  neighbours' -- the neighbourhood mean is close to a copy of the node
  itself, so aggregating over it adds nothing new. That is
  explanation (a), and it is a property of the data rather than of
  the model.
""")

# Does the neighbour mean of the target add anything beyond own features?
A = np.column_stack([np.ones(len(x_raw)), x_raw[:, 6:11]])
beta = np.linalg.lstsq(A, y_anom, rcond=None)[0]
resid = y_anom - A @ beta
nb_y = y_anom[nbr].mean(axis=1)
r_resid = float(np.corrcoef(resid, nb_y)[0, 1])
print(f"  After regressing the target on the node's OWN morphology,")
print(f"  the residual still correlates {r_resid:+.3f} with the")
print(f"  neighbourhood mean of the target.")
print("""  Large -> neighbours hold target information the features miss,
           and a graph SHOULD help (explanation b).
  Small -> the features already carry it (explanation a).

  Caveat: under spatially blocked splits a test node's neighbours are
  mostly other TEST nodes, whose targets are unavailable at inference.
  So even a large value here may not be exploitable -- that is a
  structural limit of the evaluation, not a tuning problem.
""")


# ============================================================
# PART 2 — k sweep, architecture held fixed
# ============================================================
def rebuild_loaders(D, hp, edge_index):
    out = {}
    for name in ('train', 'val', 'test'):
        d = D['pruned'][name].dataset
        ds = PhyResDataset(d.x_norm, d.x_raw, d.y, d.observed,
                           edge_index, d.pos, d.node_mask,
                           window_size=hp['window_size'],
                           horizon=hp['horizon'], use_pruning=d.use_pruning)
        out[name] = DataLoader(ds, batch_size=1, shuffle=(name == 'train'))
    return out


def eval_model(model, loader, temp_std):
    model.eval()
    P, Y, rs = [], [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            m = batch.loss_mask
            if m.sum() < 5:
                continue
            fp, _, _ = model(batch.x, batch.edge_index)
            p = fp[m].cpu().numpy(); y = batch.y[m].cpu().numpy()
            P.append(p); Y.append(y)
            pa, ya = p - p.mean(), y - y.mean()
            if pa.std() > 1e-9 and ya.std() > 1e-9:
                rs.append(np.corrcoef(ya, pa)[0, 1])
    if not P:
        return np.nan, np.nan
    p, y = np.concatenate(P), np.concatenate(Y)
    return (float(np.sqrt(((p - y) ** 2).mean()) * temp_std),
            float(np.nanmean(rs)) if rs else np.nan)


print("=" * 74)
print(f"PART 2 — k SWEEP, {CFG['city']} {CFG['month']}")
print("=" * 74)

rows = []
for k in K_GRID:
    ei = (torch.zeros((2, 0), dtype=torch.long, device=device) if k == 0
          else knn_graph(pos, k=k, loop=False).to(device))
    L = rebuild_loaders(D, HP, ei)
    for s in SEEDS:
        set_seed(s)
        model = PhyRes(len(FEATURE_COLS), HP['spatial_dim'],
                       HP['lstm_hidden'], HP['dropout']).to(device)
        model, prep = train_masked(
            model, L['train'], L['val'], HP['epochs'], lr=LR,
            alpha=ALPHA, gamma=GAMMA, weight_decay=1e-4, clip=0.5,
            tag=f"k={k} s={s}", temp_std=D['temp_std'],
            feat_mask=make_feature_mask([]))
        m = MaskedModel(model, prep).to(device)
        rmse, sr = eval_model(m, L['test'], D['temp_std'])
        rows.append(dict(k=k, seed=s, rmse=rmse, spatial_r=sr))
        del model, m
        if device.type == 'cuda':
            torch.cuda.empty_cache()
    sub = [r for r in rows if r['k'] == k]
    print(f"  k={k:<3} RMSE {np.mean([r['rmse'] for r in sub]):.3f} "
          f"(sd {np.std([r['rmse'] for r in sub]):.3f}) | "
          f"spatial r {np.mean([r['spatial_r'] for r in sub]):+.3f} "
          f"(sd {np.std([r['spatial_r'] for r in sub]):.3f})")

ks = pd.DataFrame(rows)
ks.to_csv(f"{OUT_ROOT}/k_focused.csv", index=False)

print("\n" + "=" * 74)
print("PAIRED AGAINST k=0 (identical model, no neighbours)")
print("=" * 74)
piv = ks.pivot_table(index='seed', columns='k', values='spatial_r')
pivr = ks.pivot_table(index='seed', columns='k', values='rmse')
for k in K_GRID[1:]:
    ds_ = (piv[k] - piv[0]).dropna()
    dr_ = (pivr[k] - pivr[0]).dropna()
    print(f"  k={k:<3} spatial r {ds_.mean():+.4f} (sd {ds_.std():.4f})"
          f"   RMSE {dr_.mean():+.4f} (sd {dr_.std():.4f})"
          f"   wins {int((ds_ > 0).sum())}/{len(ds_)}")

print("""
DECISION RULE — agreed before running, so the answer is not chosen
after the fact:

  k=0 ties k=8   -> the features already carry the spatial structure.
      Message passing over them is redundant. That is a finding, it
      goes in the paper, and tuning stops here. Further architecture
      search at this point would be manufacturing a result.

  k=8 clearly wins -> the earlier Full-vs-NoGraph gap was an encoder
      confound. The graph helps and stays in the paper.

  k=8 clearly loses -> something is wired wrong. Worth ONE round of
      investigation, not open-ended tuning.

Read Part 1 alongside this. If the features correlate above ~0.9 with
their neighbourhood means, redundancy is confirmed independently of
what the models do, and a tie at k=0 is the expected outcome rather
than a disappointing one.
""")

## Spatial-lag features versus message passing

In [ ]:
# ============================================================
# IS GAT THE WRONG DELIVERY MECHANISM FOR NEIGHBOURHOOD INFO?
# ------------------------------------------------------------
# WHERE THIS COMES FROM. On Delhi April, same config and fold:
#
#   NoGraph  (MLP encoder,  no graph)   RMSE 1.524
#   k=0      (GAT+skip,     no graph)   RMSE 1.765
#   k=8      (GAT+skip,     graph)      RMSE 1.724
#
# Two effects pulling opposite ways. Message passing HELPS inside the
# GAT architecture (+0.023 spatial r, 3/3 seeds, monotone in k). But
# the GAT architecture itself COSTS 0.24 RMSE against a plain MLP --
# roughly five times larger. That is why NoGraph won 5 of 6 configs
# in the full sweep.
#
# So the open question is not "do neighbours help" -- Part 1 showed
# the target residual still correlates +0.544 with the neighbourhood
# mean after regressing on own morphology, and the k-sweep showed a
# consistent gain. It is whether GAT is the right way to deliver that
# information.
#
# THE TEST. Give the MLP the same neighbourhood information without
# message passing: append the k-NN mean of each morphological feature
# as extra inputs. Spatial-lag regression, essentially, which is the
# baseline that beat plain OLS on the Landsat data.
#
#   MLP + lag beats MLP        -> neighbourhood info helps, and GAT
#       was simply a costly way to deliver it. The paper reports a
#       spatial-lag MLP and explains why message passing was dropped.
#   MLP + lag ties MLP         -> the +0.023 was GAT recovering
#       something its own architecture discarded, not new information.
#       Report the MLP and stop.
#
# 6 runs, ~35 min. This is the LAST architecture question -- beyond it
# the work becomes search rather than test.
# ============================================================
import torch
import torch.nn.functional as F          # re-import: earlier cells shadowed F
import numpy as np
import pandas as pd

CFG = next(c for c in CONFIGS if c['city'] == 'Delhi' and c['month'] == 'April')
K_LAG = 8
SEEDS = [10, 11, 12]
MORPH = [6, 7, 8, 9, 10]        # NDVI, NDBI, bld_height, bld_cover, no2

D = prepare(CFG, HP)
ds0 = D['pruned']['train'].dataset
pos = ds0.pos

# neighbour index, from the same k-NN construction the graph uses
nbr = knn_graph(pos, k=K_LAG, loop=False)[0].reshape(-1, K_LAG)

x_norm, x_raw = ds0.x_norm, ds0.x_raw
# lag features: neighbourhood mean of each morphological channel,
# per node and per timestep. Atmospherics are excluded -- ERA5 is
# already far coarser than the node spacing, so its neighbourhood
# mean is very close to a copy of itself.
lag = x_norm[nbr][:, :, :, MORPH].mean(dim=1)          # [N, T, 5]
x_aug = torch.cat([x_norm, lag], dim=2)                # [N, T, 16]
print(f"  features {x_norm.shape[2]} -> {x_aug.shape[2]} "
      f"(+{len(MORPH)} neighbourhood means at k={K_LAG})")

# how much do the lag channels actually differ from their own node?
for j, i in enumerate(MORPH):
    own = x_norm[:, -1, i].cpu().numpy()
    nb = lag[:, -1, j].cpu().numpy()
    print(f"    idx {i}: corr(own, lag) {np.corrcoef(own, nb)[0, 1]:.3f}")


def make_loaders(x_feats, hp):
    """Loaders over an arbitrary feature tensor. x_raw stays at its
    original width -- the loss indexes it by position for raw_wind
    (1:3) and bld_cover (9), so widening it would break those."""
    out = {}
    for name in ('train', 'val', 'test'):
        d = D['pruned'][name].dataset
        ds = PhyResDataset(x_feats, x_raw, d.y, d.observed,
                           d.base_edge_index, d.pos, d.node_mask,
                           window_size=hp['window_size'],
                           horizon=hp['horizon'], use_pruning=d.use_pruning)
        out[name] = DataLoader(ds, batch_size=1, shuffle=(name == 'train'))
    return out


def eval_model(model, loader, temp_std):
    model.eval()
    P, Y, rs = [], [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            m = batch.loss_mask
            if m.sum() < 5:
                continue
            fp, _, _ = model(batch.x, batch.edge_index)
            p = fp[m].cpu().numpy(); y = batch.y[m].cpu().numpy()
            P.append(p); Y.append(y)
            pa, ya = p - p.mean(), y - y.mean()
            if pa.std() > 1e-9 and ya.std() > 1e-9:
                rs.append(np.corrcoef(ya, pa)[0, 1])
    if not P:
        return np.nan, np.nan
    p, y = np.concatenate(P), np.concatenate(Y)
    return (float(np.sqrt(((p - y) ** 2).mean()) * temp_std),
            float(np.nanmean(rs)) if rs else np.nan)


ARMS = [
    # (label, feature tensor, n_features)
    ('MLP',       x_norm, x_norm.shape[2]),
    ('MLP + lag', x_aug,  x_aug.shape[2]),
]

rows = []
print("\n" + "=" * 74)
print(f"MLP vs MLP + SPATIAL LAG — {CFG['city']} {CFG['month']}")
print("=" * 74)

for label, feats, nf in ARMS:
    L = make_loaders(feats, HP)
    # mask must match the feature width; nothing is dropped here
    fmask = torch.ones(1, 1, nf, device=device)
    for s in SEEDS:
        set_seed(s)
        # n_atmos/n_urban keep the head slices at 0:6 and 6:11, so the
        # appended lag channels feed the encoder and LSTM only -- the
        # physics and residual heads see exactly what they did before.
        model = NoGraphModel(nf, HP['spatial_dim'], HP['lstm_hidden'],
                             HP['dropout'], n_atmos=6, n_urban=5).to(device)
        model, prep = train_masked(
            model, L['train'], L['val'], HP['epochs'], lr=LR,
            alpha=ALPHA, gamma=GAMMA, weight_decay=1e-4, clip=0.5,
            tag=f"{label} s={s}", temp_std=D['temp_std'], feat_mask=fmask)
        m = MaskedModel(model, prep).to(device)
        rmse, sr = eval_model(m, L['test'], D['temp_std'])
        rows.append(dict(arm=label, seed=s, rmse=rmse, spatial_r=sr))
        del model, m
        if device.type == 'cuda':
            torch.cuda.empty_cache()
    sub = [r for r in rows if r['arm'] == label]
    print(f"  {label:<11} RMSE {np.mean([r['rmse'] for r in sub]):.3f} "
          f"(sd {np.std([r['rmse'] for r in sub]):.3f}) | "
          f"spatial r {np.mean([r['spatial_r'] for r in sub]):+.3f} "
          f"(sd {np.std([r['spatial_r'] for r in sub]):.3f})")

res = pd.DataFrame(rows)
res.to_csv(f"{OUT_ROOT}/mlp_lag_test.csv", index=False)

piv_r = res.pivot_table(index='seed', columns='arm', values='spatial_r')
piv_e = res.pivot_table(index='seed', columns='arm', values='rmse')
dr = (piv_r['MLP + lag'] - piv_r['MLP']).dropna()
de = (piv_e['MLP + lag'] - piv_e['MLP']).dropna()

print("\n" + "=" * 74)
print("PAIRED, MLP + lag MINUS MLP")
print("=" * 74)
print(f"  spatial r {dr.mean():+.4f} (sd {dr.std():.4f})  "
      f"wins {int((dr > 0).sum())}/{len(dr)}")
print(f"  RMSE      {de.mean():+.4f} (sd {de.std():.4f})  "
      f"wins {int((de < 0).sum())}/{len(de)}")

print(f"""
FOR REFERENCE, same config and fold, from earlier runs:
  GAT+skip, no graph  (k=0)   RMSE 1.765   spatial r 0.686
  GAT+skip, k=8               RMSE 1.724   spatial r 0.709
  MLP, no graph (cell 9)      RMSE 1.524

READING IT
  If MLP + lag improves on MLP by roughly the same margin the graph
  gave GAT (+0.023 spatial r), then neighbourhood information helps
  and message passing was an expensive way to deliver it. The honest
  paper reports a spatial-lag MLP, states that a GAT was tried, and
  explains why it was dropped -- the encoder cost outweighed the
  message-passing gain by about five to one.

  If MLP + lag ties MLP, the GAT's k-gain was recovering structure its
  own aggregation had discarded, not adding information. Report the
  plain MLP.

  Either way this is a SINGLE CONFIG, one fold, three seeds. State it
  as such. The 14-day sweep varied by config -- Bengaluru April
  favoured k=16, Hyderabad December k=8, Bengaluru December was flat
  -- so a second config would be the first thing a reviewer asks for.
""")

## Coverage diagnostic — usable night passes per configuration

In [ ]:
# ============================================================
# COVERAGE DIAGNOSTIC — how many usable night passes are there?
# ------------------------------------------------------------
# Bengaluru April has 13 night passes but only 2 with >=60% node
# coverage. MODIS LST is a clear-sky product, so a partly cloudy
# night yields a partly covered scene, and pre-monsoon Bengaluru is
# cloudy at night.
#
# The choice is between:
#   (a) lowering MIN_TIME_COVERAGE  -- keeps more passes, imputes more
#   (b) shortening window_size      -- fewer passes needed per sample
#   (c) re-extracting a longer date range -- more real passes
#
# (c) is the only one that adds information. (a) and (b) redistribute
# what is already there. This prints what each would buy, per config,
# so the decision is made on numbers rather than preference.
#
# SELF-CONTAINED apart from CONFIGS / ID_COL / TIME_COL.
# ============================================================
import numpy as np
import pandas as pd

COV_GRID = [0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1]
WINDOWS = [3, 5, 8]

rows = []
print("=" * 78)
print("NIGHT-PASS COVERAGE BY CONFIG")
print("=" * 78)

for cfg in CONFIGS:
    tag = f"{cfg['short']} {cfg['month']}"
    df = pd.read_csv(cfg['csv'])
    df[ID_COL] = _align_ids(df[ID_COL])
    if np.issubdtype(df[TIME_COL].dtype, np.number):
        df[TIME_COL] = pd.to_datetime(df[TIME_COL], unit='ms')
    else:
        df[TIME_COL] = pd.to_datetime(df[TIME_COL])

    night = df[df['pass'].str.contains('night', na=False)]
    n_nodes = night[ID_COL].nunique()
    n_pass = night[TIME_COL].nunique()
    valid = night[night['lst_modis'].notna()]

    # fraction of nodes with a valid retrieval, per night pass
    cov = valid.groupby(TIME_COL)[ID_COL].nunique() / n_nodes
    cov = cov.reindex(sorted(night[TIME_COL].unique()), fill_value=0.0)

    print(f"\n{tag}   {n_nodes} nodes, {n_pass} night passes, "
          f"{valid['lst_modis'].notna().sum() / (n_nodes * n_pass):.0%} of "
          f"cells observed")
    print("  per-pass node coverage, sorted:")
    print("    " + "  ".join(f"{c:.2f}" for c in sorted(cov.values, reverse=True)))

    print(f"  {'threshold':>10}{'passes kept':>13}"
          + "".join(f"{'win ' + str(w):>10}" for w in WINDOWS))
    for t in COV_GRID:
        kept = int((cov >= t).sum())
        cells = []
        for w in WINDOWS:
            # windows available across a 60/15/25 split
            tr = int(kept * 0.60) - w + 1
            va = max(1, int(kept * 0.15)) - w + 1
            te = kept - int(kept * 0.60) - max(1, int(kept * 0.15)) - w + 1
            ok = min(tr, va, te)
            cells.append(f"{max(ok,0):>10}")
        print(f"  {t:>10.1f}{kept:>13}" + "".join(cells))
        rows.append(dict(config=tag, threshold=t, passes=kept,
                         **{f"win{w}": None for w in WINDOWS}))

    print("  (last three columns: MINIMUM windows across train/val/test.")
    print("   Below ~3 in any split, that configuration cannot train.)")

print("\n" + "=" * 78)
print("WHAT THIS MEANS")
print("=" * 78)
print("""
  If a config needs a threshold below ~0.3 to get enough passes, most
  of its target matrix is interpolated and any result from it is
  largely a statement about the interpolation.

  A 14-day window gives at most ~28 night passes (Terra + Aqua, one
  each per day) before cloud. Extending to 60 days gives ~120, so
  even at 30% clear-sky retention you would have ~36 usable passes --
  more than any threshold choice can recover from 14 days.

  The cost of a longer window is that surface conditions drift: NDVI
  and the Landsat scene are single snapshots, and over 60 days
  vegetation changes. That is a real limitation to state, but a
  smaller one than training on two timestamps.

  Recommended if several configs look thin: re-extract with
  ['2025-03-01','2025-04-30'] and ['2025-11-15','2026-01-15'],
  keeping the same Landsat target dates. Same script, two dates
  changed per run.
""")

## MODIS preflight — standalone sanity check on the extracts

In [ ]:
# ============================================================
# MODIS PREFLIGHT — standalone, cold kernel
# ------------------------------------------------------------
# Mounts Drive, stages locally, and validates the new 1 km MODIS
# extraction. Assumes NOTHING is defined. Run this cell alone.
#
# The decisive test is Block D. ERA5 failed as a target because its
# spatial standard deviation across nodes was 0.09-0.23 C against ~4 C
# temporal -- about 3% of the variance was spatial, and even that was
# the shape of its own 11 km grid rather than urban structure. If
# MODIS night LST does not show clearly more spatial variance than
# that, this target is no better and no architecture will fix it.
#
# Everything before Block D is structure. Block D is the answer.
# ============================================================
import os
import shutil
import numpy as np
import pandas as pd
import geopandas as gpd

# ---- mount + stage ------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATA = '/content/drive/MyDrive/UHI'
LOCAL_DATA = '/content/data_modis'

DATASETS = [
    ('Bangalore', 'Blr', 'April',    'Blr_April_MODIS_v2.csv',    'Blr_Nodes_1km.geojson'),
    ('Bangalore', 'Blr', 'December', 'Blr_December_MODIS_v2.csv', 'Blr_Nodes_1km.geojson'),
    ('Hyderabad', 'Hyd', 'April',    'Hyd_April_MODIS_v2.csv',    'Hyd_Nodes_1km.geojson'),
    ('Hyderabad', 'Hyd', 'December', 'Hyd_December_MODIS_v2.csv', 'Hyd_Nodes_1km.geojson'),
    ('Delhi',     'Dlh', 'April',    'Dlh_April_MODIS_v2.csv',    'Dlh_Nodes_1km.geojson'),
    ('Delhi',     'Dlh', 'December', 'Dlh_December_MODIS_v2.csv', 'Dlh_Nodes_1km.geojson'),
]

os.makedirs(LOCAL_DATA, exist_ok=True)
print(f"Staging to {LOCAL_DATA}:")
for name in sorted({f for d in DATASETS for f in d[3:5]}):
    src, dst = f"{DRIVE_DATA}/{name}", f"{LOCAL_DATA}/{name}"
    if not os.path.exists(src):
        print(f"  [MISSING ON DRIVE] {name}")
        continue
    if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
        print(f"  [cached ] {name}  ({os.path.getsize(dst)/1e6:.1f} MB)")
        continue
    shutil.copy2(src, dst)
    print(f"  [staged ] {name}  ({os.path.getsize(dst)/1e6:.1f} MB)")

if os.path.isdir(DRIVE_DATA):
    have = set(os.listdir(LOCAL_DATA))
    want = {f for d in DATASETS for f in d[3:5]}
    if want - have:
        print(f"\n  Drive contains: {sorted(os.listdir(DRIVE_DATA))}")

CONFIGS_M = [dict(city=c, short=s, month=m,
                  csv=f'{LOCAL_DATA}/{csv}', geojson=f'{LOCAL_DATA}/{geo}')
             for c, s, m, csv, geo in DATASETS]

ID_COL, TIME_COL = 'id', 'timestamp'
TARGET = 'lst_modis'
STATIC = ['bld_cover', 'bld_height', 'NDVI', 'NDBI', 'lst_landsat']
ATMOS = ['t2m', 'u10', 'v10', 'ssr', 'str', 'slhf', 'lai_hi', 'lai_lo']
EXPECTED_MONTH = {'April': 4, 'December': 12}

summary = []

for cfg in CONFIGS_M:
    tag = f"{cfg['short']}_{cfg['month']}"
    print("\n" + "=" * 70)
    print(f"{tag}   ({cfg['city']})")
    print("=" * 70)

    if not os.path.exists(cfg['csv']):
        print("  MISSING CSV\n")
        summary.append((tag, 'MISSING', None, None, None))
        continue

    df = pd.read_csv(cfg['csv'])
    missing = [c for c in [TARGET, 'pass', ID_COL, TIME_COL] if c not in df.columns]
    if missing:
        print(f"  MISSING COLUMNS {missing}")
        print(f"  present: {list(df.columns)}\n")
        summary.append((tag, 'BAD SCHEMA', None, None, None))
        continue

    # ---- A. STRUCTURE ---------------------------------------
    if np.issubdtype(df[TIME_COL].dtype, np.number):
        df[TIME_COL] = pd.to_datetime(df[TIME_COL], unit='ms')
    else:
        df[TIME_COL] = pd.to_datetime(df[TIME_COL])

    n_nodes = df[ID_COL].nunique()
    n_obs = df[TIME_COL].nunique()
    print(f"A. rows {len(df):,} | nodes {n_nodes:,} | timestamps {n_obs}")
    print(f"   span {df[TIME_COL].min()} -> {df[TIME_COL].max()}")
    months = sorted(int(m) for m in df[TIME_COL].dt.month.unique())
    print(f"   months {months} (expected {EXPECTED_MONTH[cfg['month']]})"
          f"{'  OK' if months == [EXPECTED_MONTH[cfg['month']]] else '  <-- CHECK'}")

    if os.path.exists(cfg['geojson']):
        g = gpd.read_file(cfg['geojson'])
        print(f"   geojson nodes {len(g):,}"
              + ("" if len(g) == n_nodes else "  <-- MISMATCH"))

    print("\n   observations per pass type:")
    for p, k in df.groupby('pass')[TIME_COL].nunique().items():
        print(f"     {p:<14}{k:>4} timestamps")

    # ---- B. COVERAGE / CLOUD --------------------------------
    valid = df[TARGET].notna()
    print(f"\nB. valid LST retrievals: {valid.sum():,}/{len(df):,} "
          f"({valid.mean():.1%})")
    if 'qc_ok' in df.columns:
        print(f"   flagged 'good' quality: {df['qc_ok'].mean():.1%} of rows")
    per_node = df.groupby(ID_COL)[TARGET].apply(lambda s: s.notna().sum())
    print(f"   valid obs per node: min {per_node.min()}  "
          f"median {int(per_node.median())}  max {per_node.max()}")
    if per_node.median() < 15:
        print("   !! median under 15 -- a 12-step LSTM window will not fit")
        print("      many nodes. Cloud masking may be too aggressive, or")
        print("      the window too short.")

    # ---- C. STATIC COLUMNS ----------------------------------
    have_static = [c for c in STATIC if c in df.columns]
    node_tab = df.groupby(ID_COL)[have_static].mean()
    print("\nC. node-level static features:")
    print(node_tab.describe().round(4).to_string().replace('\n', '\n   '))
    if 'bld_cover' in node_tab:
        mx = node_tab['bld_cover'].max()
        print(f"\n   bld_cover max {mx:.3f} -- at 1 km this SHOULD be lower")
        print(f"   than the 500 m grid's 0.59-0.69, since dense cores get")
        print(f"   averaged with their surroundings. A very low MEDIAN "
              f"({node_tab['bld_cover'].median():.3f}) would be the problem.")

    # ---- D. THE DECISIVE TEST -------------------------------
    print("\nD. SPATIAL vs TEMPORAL VARIANCE  <-- this decides everything")
    for pass_name in sorted(df['pass'].dropna().unique()):
        sub = df[(df['pass'] == pass_name) & df[TARGET].notna()]
        if len(sub) < 50:
            print(f"   {pass_name:<14} too few valid rows")
            continue
        # spatial: std across nodes within a timestamp, averaged
        spatial = sub.groupby(TIME_COL)[TARGET].std().mean()
        # temporal: std of the field mean across timestamps
        temporal = sub.groupby(TIME_COL)[TARGET].mean().std()
        ratio = spatial / temporal if temporal > 0 else np.nan
        print(f"   {pass_name:<14} spatial {spatial:>5.2f} C | "
              f"temporal {temporal:>5.2f} C | ratio {ratio:>5.2f}")

    night = df[df['pass'].str.contains('night', na=False) & df[TARGET].notna()]
    night_spatial = (night.groupby(TIME_COL)[TARGET].std().mean()
                     if len(night) else np.nan)
    print(f"\n   NIGHT spatial std = {night_spatial:.2f} C")
    print(f"   ERA5 gave 0.09-0.23 C and that is what killed the last design.")
    if np.isnan(night_spatial):
        verdict = 'NO NIGHT DATA'
    elif night_spatial >= 1.0:
        verdict = 'GOOD TARGET'
        print("   -> GOOD. Real intra-urban structure to learn.")
    elif night_spatial >= 0.5:
        verdict = 'MARGINAL'
        print("   -> MARGINAL. Some structure, but weak. Check the decile")
        print("      table against bld_cover before committing.")
    else:
        verdict = 'TOO FLAT'
        print("   -> TOO FLAT. No better than ERA5. Do not build on this.")

    # does the spatial pattern relate to urban form?
    if 'bld_cover' in node_tab and len(night):
        nm = night.groupby(ID_COL)[TARGET].mean()
        j = pd.concat([nm.rename('lst'), node_tab['bld_cover']], axis=1).dropna()
        r_cover = j['lst'].corr(j['bld_cover'])
        print(f"\n   corr(night LST, bld_cover) = {r_cover:+.3f}")
        print("   Nocturnal UHI predicts POSITIVE: built mass releases stored")
        print("   heat after dark. The Landsat 10:30 daytime analysis gave")
        print("   -0.04 to +0.50 depending on city and season. A consistently")
        print("   positive value here across all six configs is the result")
        print("   the original push-pull prior always assumed.")
    else:
        r_cover = np.nan

    summary.append((tag, verdict, night_spatial, r_cover,
                    int(per_node.median())))

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
print(f"  {'config':<16}{'verdict':<15}{'night sd':>9}"
      f"{'r(LST,cover)':>14}{'obs/node':>10}")
for tag, v, sd, r, n in summary:
    print(f"  {tag:<16}{v:<15}"
          f"{(f'{sd:.2f}' if sd is not None and not np.isnan(sd) else '-'):>9}"
          f"{(f'{r:+.3f}' if r is not None and not np.isnan(r) else '-'):>14}"
          f"{(str(n) if n else '-'):>10}")

good = sum(1 for _, v, _, _, _ in summary if v == 'GOOD TARGET')
print(f"\n  {good}/{len(summary)} configs have a usable target.")
print("""
  If night spatial std is above ~1 C and corr(night LST, bld_cover) is
  consistently positive, the original design is back on solid ground:
  a target with genuine intra-urban structure, sampled often enough
  for the temporal model, and a physics prior pointing the right way.

  If it is flat, stop here. That is the same failure as ERA5 and no
  amount of architecture will change it.
""")